# Phase 6: Live NBA Prediction Pipeline

## Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

## Loading Historical Dataset

In [2]:
df = pd.read_csv(
    "../data/processed/team_game_modeling.csv",
    dtype={
        "GAME_ID": "string",
        "SEASON": "string",
        "TEAM_ID": "Int64",
    },
    parse_dates=["GAME_DATE"],
)

print(df.shape)
df.head()

(12300, 77)


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,OPP_REST_DAYS,PTS_ROLL5_DIFF,FG_PCT_ROLL5_DIFF,FG3_PCT_ROLL5_DIFF,REB_ROLL5_DIFF,AST_ROLL5_DIFF,TOV_ROLL5_DIFF,PLUS_MINUS_ROLL5_DIFF,WIN_PCT_ROLL5_DIFF,REST_DAYS_DIFF
0,22021,1610612737,ATL,Atlanta Hawks,0022100014,2021-10-21,ATL vs. DAL,W,240,45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22021,1610612737,ATL,Atlanta Hawks,0022100027,2021-10-23,ATL @ CLE,L,240,38,...,1.0,-3.50,-0.02950,0.10000,19.000000,-1.0,-2.000000,37.000000,1.000000,1.0
2,22021,1610612737,ATL,Atlanta Hawks,0022100043,2021-10-25,ATL vs. DET,W,240,46,...,2.0,19.00,0.03850,0.16500,4.500000,10.0,-9.000000,20.500000,0.500000,0.0
3,22021,1610612737,ATL,Atlanta Hawks,0022100059,2021-10-27,ATL @ NOP,W,240,40,...,2.0,8.75,0.03350,-0.01100,0.916667,3.0,-7.583333,21.166667,0.416667,0.0
4,22021,1610612737,ATL,Atlanta Hawks,0022100066,2021-10-28,ATL @ WAS,L,240,48,...,1.0,-1.75,-0.00825,0.02225,6.000000,4.5,-2.500000,7.500000,0.000000,0.0


In [3]:
df["SEASON"].value_counts().sort_index()

SEASON
2021-22    2460
2022-23    2460
2023-24    2460
2024-25    2460
2025-26    2460
Name: count, dtype: Int64

In [4]:
features = [
    "HOME_GAME",
    "REST_DAYS",
    "SEASON_WIN_PCT",
    "TEAM_GAME_NUMBER",
    "PTS_ROLL5",
    "FG_PCT_ROLL5",
    "FG3_PCT_ROLL5",
    "FT_PCT_ROLL5",
    "REB_ROLL5",
    "AST_ROLL5",
    "STL_ROLL5",
    "BLK_ROLL5",
    "TOV_ROLL5",
    "PLUS_MINUS_ROLL5",
    "WIN_PCT_ROLL5",
    "OPP_PTS_ROLL5",
    "OPP_FG_PCT_ROLL5",
    "OPP_FG3_PCT_ROLL5",
    "OPP_FT_PCT_ROLL5",
    "OPP_REB_ROLL5",
    "OPP_AST_ROLL5",
    "OPP_STL_ROLL5",
    "OPP_BLK_ROLL5",
    "OPP_TOV_ROLL5",
    "OPP_PLUS_MINUS_ROLL5",
    "OPP_WIN_PCT_ROLL5",
    "OPP_REST_DAYS",
    "PTS_ROLL5_DIFF",
    "FG_PCT_ROLL5_DIFF",
    "FG3_PCT_ROLL5_DIFF",
    "REB_ROLL5_DIFF",
    "AST_ROLL5_DIFF",
    "TOV_ROLL5_DIFF",
    "PLUS_MINUS_ROLL5_DIFF",
    "WIN_PCT_ROLL5_DIFF",
    "REST_DAYS_DIFF",
]

target = "WIN"

In [5]:
production_df = (
    df.dropna(
        subset=features + [target]
    )
    .copy()
)

X_production = production_df[features]
y_production = production_df[target]

print("Production rows:", len(production_df))
print("Features:", X_production.shape[1])

Production rows: 12142
Features: 36


## Model Recreation

In [6]:
production_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                random_state=42,
                max_iter=1000
            )
        )
    ]
)

production_model.fit(
    X_production,
    y_production
)

print("Production model trained.")

Production model trained.


In [7]:
production_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                random_state=42,
                max_iter=1000
            )
        )
    ]
)

production_model.fit(
    X_production,
    y_production
)

print("Production model trained.")

Production model trained.


## Predictions Setup

In [8]:
latest_team_state = (
    df.sort_values("GAME_DATE")
    .groupby("TEAM_NAME")
    .tail(1)
    .copy()
)

In [9]:
team_state_columns = [
    "TEAM_NAME",
    "TEAM_ID",
    "GAME_DATE",
    "SEASON_WIN_PCT",
    "TEAM_GAME_NUMBER",
    "PTS_ROLL5",
    "FG_PCT_ROLL5",
    "FG3_PCT_ROLL5",
    "FT_PCT_ROLL5",
    "REB_ROLL5",
    "AST_ROLL5",
    "STL_ROLL5",
    "BLK_ROLL5",
    "TOV_ROLL5",
    "PLUS_MINUS_ROLL5",
    "WIN_PCT_ROLL5",
]

latest_team_state = latest_team_state[
    team_state_columns
].reset_index(drop=True)

latest_team_state.head()

,TEAM_NAME,TEAM_ID,GAME_DATE,SEASON_WIN_PCT,TEAM_GAME_NUMBER,PTS_ROLL5,FG_PCT_ROLL5,FG3_PCT_ROLL5,FT_PCT_ROLL5,REB_ROLL5,AST_ROLL5,STL_ROLL5,BLK_ROLL5,TOV_ROLL5,PLUS_MINUS_ROLL5,WIN_PCT_ROLL5
0,Brooklyn Nets,1610612751,2026-04-12,0.246914,82,105.2,0.4520,0.3522,0.8280,36.0,23.8,8.2,4.8,13.2,-13.6,0.4
1,Washington Wizards,1610612764,2026-04-12,0.209877,82,114.8,0.4710,0.3578,0.7310,42.0,24.0,10.4,5.0,17.4,-17.4,0.0
2,Atlanta Hawks,1610612737,2026-04-12,0.567901,82,123.2,0.4884,0.4032,0.7444,45.8,27.2,10.6,5.8,14.2,15.2,0.6
3,Boston Celtics,1610612738,2026-04-12,0.679012,82,122.2,0.5168,0.4028,0.8192,44.6,27.6,7.2,3.0,13.2,15.4,0.8
4,Cleveland Cavaliers,1610612739,2026-04-12,0.629630,82,120.2,0.4964,0.3564,0.7856,45.4,27.4,8.2,4.8,13.6,3.2,0.8


In [10]:
print(
    "Teams:",
    latest_team_state["TEAM_NAME"].nunique()
)

Teams: 30


In [11]:
def get_latest_team_state(
    team_name,
    team_state_df
):
    team_rows = team_state_df[
        team_state_df["TEAM_NAME"]
        == team_name
    ]

    if team_rows.empty:
        raise ValueError(
            f"No team state found for {team_name}"
        )

    return team_rows.iloc[0]

In [12]:
get_latest_team_state(
    "Boston Celtics",
    latest_team_state
)

TEAM_NAME                Boston Celtics
TEAM_ID                      1610612738
GAME_DATE           2026-04-12 00:00:00
SEASON_WIN_PCT                 0.679012
TEAM_GAME_NUMBER                     82
PTS_ROLL5                         122.2
FG_PCT_ROLL5                     0.5168
FG3_PCT_ROLL5                    0.4028
FT_PCT_ROLL5                     0.8192
REB_ROLL5                          44.6
AST_ROLL5                          27.6
STL_ROLL5                           7.2
BLK_ROLL5                           3.0
TOV_ROLL5                          13.2
PLUS_MINUS_ROLL5                   15.4
WIN_PCT_ROLL5                       0.8
Name: 3, dtype: object

In [13]:
def build_live_matchup_row(
    team_row,
    opponent_row,
    home_game,
    team_rest_days,
    opponent_rest_days,
    team_game_number
):
    matchup = pd.DataFrame(index=[0])

    matchup["HOME_GAME"] = [home_game]
    matchup["REST_DAYS"] = [team_rest_days]
    matchup["SEASON_WIN_PCT"] = [
        team_row["SEASON_WIN_PCT"]
    ]
    matchup["TEAM_GAME_NUMBER"] = [
        team_game_number
    ]

    team_stats = [
        "PTS_ROLL5",
        "FG_PCT_ROLL5",
        "FG3_PCT_ROLL5",
        "FT_PCT_ROLL5",
        "REB_ROLL5",
        "AST_ROLL5",
        "STL_ROLL5",
        "BLK_ROLL5",
        "TOV_ROLL5",
        "PLUS_MINUS_ROLL5",
        "WIN_PCT_ROLL5",
    ]

    for stat in team_stats:
        matchup[stat] = [
            team_row[stat]
        ]

        matchup[
            f"OPP_{stat}"
        ] = [
            opponent_row[stat]
        ]

    matchup["OPP_REST_DAYS"] = [
        opponent_rest_days
    ]

    matchup["PTS_ROLL5_DIFF"] = (
        matchup["PTS_ROLL5"]
        - matchup["OPP_PTS_ROLL5"]
    )

    matchup["FG_PCT_ROLL5_DIFF"] = (
        matchup["FG_PCT_ROLL5"]
        - matchup["OPP_FG_PCT_ROLL5"]
    )

    matchup["FG3_PCT_ROLL5_DIFF"] = (
        matchup["FG3_PCT_ROLL5"]
        - matchup["OPP_FG3_PCT_ROLL5"]
    )

    matchup["REB_ROLL5_DIFF"] = (
        matchup["REB_ROLL5"]
        - matchup["OPP_REB_ROLL5"]
    )

    matchup["AST_ROLL5_DIFF"] = (
        matchup["AST_ROLL5"]
        - matchup["OPP_AST_ROLL5"]
    )

    matchup["TOV_ROLL5_DIFF"] = (
        matchup["TOV_ROLL5"]
        - matchup["OPP_TOV_ROLL5"]
    )

    matchup["PLUS_MINUS_ROLL5_DIFF"] = (
        matchup["PLUS_MINUS_ROLL5"]
        - matchup[
            "OPP_PLUS_MINUS_ROLL5"
        ]
    )

    matchup["WIN_PCT_ROLL5_DIFF"] = (
        matchup["WIN_PCT_ROLL5"]
        - matchup[
            "OPP_WIN_PCT_ROLL5"
        ]
    )

    matchup["REST_DAYS_DIFF"] = (
        matchup["REST_DAYS"]
        - matchup["OPP_REST_DAYS"]
    )

    return matchup[features]

## Testing Future Prediction

In [14]:
home_team = "Boston Celtics"
away_team = "Oklahoma City Thunder"

In [15]:
home_state = get_latest_team_state(
    home_team,
    latest_team_state
)

away_state = get_latest_team_state(
    away_team,
    latest_team_state
)

In [16]:
matchup = build_live_matchup_row(
    team_row=home_state,
    opponent_row=away_state,
    home_game=1,
    team_rest_days=2,
    opponent_rest_days=1,
    team_game_number=1
)

In [17]:
home_win_probability = (
    production_model.predict_proba(
        matchup
    )[0, 1]
)

print(
    f"{home_team} Win Probability: "
    f"{home_win_probability:.1%}"
)

print(
    f"{away_team} Win Probability: "
    f"{1 - home_win_probability:.1%}"
)

Boston Celtics Win Probability: 52.5%
Oklahoma City Thunder Win Probability: 47.5%


## Early-Season Initialization

At the beginning of a new NBA season, current-season rolling statistics are not yet available.

The prediction pipeline therefore initializes team state using the final available statistics from the previous season. As new games are played, these prior-season values can gradually be replaced with current-season information.

This prevents missing early-season features while allowing the model to transition toward current performance as more data becomes available.

## Prior/Current Season Build

In [18]:
def blend_feature(
    prior_value,
    current_value,
    games_played,
    transition_games=10
):
    weight_current = min(
        games_played / transition_games,
        1.0
    )

    weight_prior = 1 - weight_current

    return (
        prior_value * weight_prior
        + current_value * weight_current
    )

In [19]:
schedule = pd.DataFrame(
    {
        "GAME_DATE": [
            "2026-10-20",
            "2026-10-20"
        ],
        "HOME_TEAM": [
            "Boston Celtics",
            "Los Angeles Lakers"
        ],
        "AWAY_TEAM": [
            "Oklahoma City Thunder",
            "Golden State Warriors"
        ]
    }
)

schedule["GAME_DATE"] = pd.to_datetime(
    schedule["GAME_DATE"]
)

#### ^^ Later to be automated using NBA API

In [20]:
def calculate_rest_days(
    team,
    game_date,
    schedule
):
    previous_games = schedule[
        (
            (
                schedule["HOME_TEAM"] == team
            )
            |
            (
                schedule["AWAY_TEAM"] == team
            )
        )
        &
        (
            schedule["GAME_DATE"]
            < game_date
        )
    ]

    if previous_games.empty:
        return 3

    previous_game_date = (
        previous_games[
            "GAME_DATE"
        ].max()
    )

    return max(
        (
            game_date
            - previous_game_date
        ).days - 1,
        0
    )

## Game Prediction Function

In [21]:
def predict_game(
    home_team,
    away_team,
    game_date,
    schedule,
    team_state_df,
    model,
    team_game_number=1
):
    home_state = get_latest_team_state(
        home_team,
        team_state_df
    )

    away_state = get_latest_team_state(
        away_team,
        team_state_df
    )

    home_rest = calculate_rest_days(
        home_team,
        game_date,
        schedule
    )

    away_rest = calculate_rest_days(
        away_team,
        game_date,
        schedule
    )

    matchup = build_live_matchup_row(
        team_row=home_state,
        opponent_row=away_state,
        home_game=1,
        team_rest_days=home_rest,
        opponent_rest_days=away_rest,
        team_game_number=team_game_number
    )

    home_probability = (
        model.predict_proba(
            matchup
        )[0, 1]
    )

    return {
        "Game Date": game_date,
        "Home Team": home_team,
        "Away Team": away_team,
        "Home Win Probability": home_probability,
        "Away Win Probability": 1 - home_probability
    }

In [22]:
def predict_schedule(
    schedule,
    team_state_df,
    model
):
    predictions = []

    for _, game in schedule.iterrows():
        prediction = predict_game(
            home_team=game["HOME_TEAM"],
            away_team=game["AWAY_TEAM"],
            game_date=game["GAME_DATE"],
            schedule=schedule,
            team_state_df=team_state_df,
            model=model
        )

        predictions.append(
            prediction
        )

    return pd.DataFrame(
        predictions
    )

In [23]:
daily_predictions = predict_schedule(
    schedule,
    latest_team_state,
    production_model
)

daily_predictions

,Game Date,Home Team,Away Team,Home Win Probability,Away Win Probability
0,2026-10-20,Boston Celtics,Oklahoma City Thunder,0.494033,0.505967
1,2026-10-20,Los Angeles Lakers,Golden State Warriors,0.616559,0.383441


In [24]:
daily_predictions[
    "Predicted Winner"
] = np.where(
    daily_predictions[
        "Home Win Probability"
    ] >= 0.5,
    daily_predictions["Home Team"],
    daily_predictions["Away Team"]
)

In [25]:
daily_predictions[
    "Prediction Confidence"
] = np.maximum(
    daily_predictions[
        "Home Win Probability"
    ],
    daily_predictions[
        "Away Win Probability"
    ]
)

In [26]:
daily_predictions.sort_values(
    "Prediction Confidence",
    ascending=False
)

,Game Date,Home Team,Away Team,Home Win Probability,Away Win Probability,Predicted Winner,Prediction Confidence
1,2026-10-20,Los Angeles Lakers,Golden State Warriors,0.616559,0.383441,Los Angeles Lakers,0.616559
0,2026-10-20,Boston Celtics,Oklahoma City Thunder,0.494033,0.505967,Oklahoma City Thunder,0.505967


In [27]:
daily_predictions.to_csv(
    "../data/processed/latest_predictions.csv",
    index=False
)

## Validation

In [28]:
print(
    daily_predictions[
        [
            "Home Win Probability",
            "Away Win Probability"
        ]
    ].sum(axis=1).head()
)

0    1.0
1    1.0
dtype: float64


In [29]:
daily_predictions.isna().sum()

Game Date                0
Home Team                0
Away Team                0
Home Win Probability     0
Away Win Probability     0
Predicted Winner         0
Prediction Confidence    0
dtype: int64

In [30]:
schedule = pd.read_csv(
    "../data/raw/upcoming_schedule.csv",
    parse_dates=["GAME_DATE"]
)

schedule = schedule.sort_values("GAME_DATE").reset_index(drop=True)

schedule.head()

,GAME_DATE,HOME_TEAM,AWAY_TEAM
0,2026-10-20,Boston Celtics,Oklahoma City Thunder
1,2026-10-20,Los Angeles Lakers,Golden State Warriors
2,2026-10-21,New York Knicks,Miami Heat


## Upcoming Schedule

Upcoming games are supplied manually in this version of the pipeline. Each row contains the game date, home team, and away team.

The remainder of the prediction pipeline is independent of the schedule source, allowing this manual input to be replaced by automated NBA schedule ingestion in a future version.

In [31]:
known_teams = set(latest_team_state["TEAM_NAME"])

schedule_teams = set(
    schedule["HOME_TEAM"]
).union(
    set(schedule["AWAY_TEAM"])
)

unknown_teams = schedule_teams - known_teams

print("Unknown teams:", unknown_teams)

Unknown teams: set()


## Prediction Deliverable

In [32]:
def calculate_rest_days(
    team,
    game_date,
    schedule,
    default_rest=3
):
    previous_games = schedule[
        (
            (schedule["HOME_TEAM"] == team)
            |
            (schedule["AWAY_TEAM"] == team)
        )
        &
        (schedule["GAME_DATE"] < game_date)
    ]

    if previous_games.empty:
        return default_rest

    previous_game_date = previous_games[
        "GAME_DATE"
    ].max()

    return max(
        (game_date - previous_game_date).days - 1,
        0
    )

In [33]:
def calculate_team_game_number(
    team,
    game_date,
    schedule
):
    prior_games = schedule[
        (
            (schedule["HOME_TEAM"] == team)
            |
            (schedule["AWAY_TEAM"] == team)
        )
        &
        (schedule["GAME_DATE"] < game_date)
    ]

    return len(prior_games) + 1

In [34]:
def predict_game(
    home_team,
    away_team,
    game_date,
    schedule,
    team_state_df,
    model
):
    home_state = get_latest_team_state(
        home_team,
        team_state_df
    )

    away_state = get_latest_team_state(
        away_team,
        team_state_df
    )

    home_rest = calculate_rest_days(
        home_team,
        game_date,
        schedule
    )

    away_rest = calculate_rest_days(
        away_team,
        game_date,
        schedule
    )

    home_game_number = calculate_team_game_number(
        home_team,
        game_date,
        schedule
    )

    matchup = build_live_matchup_row(
        team_row=home_state,
        opponent_row=away_state,
        home_game=1,
        team_rest_days=home_rest,
        opponent_rest_days=away_rest,
        team_game_number=home_game_number
    )

    home_probability = model.predict_proba(
        matchup
    )[0, 1]

    away_probability = 1 - home_probability

    predicted_winner = (
        home_team
        if home_probability >= 0.5
        else away_team
    )

    confidence = max(
        home_probability,
        away_probability
    )

    return {
        "Game Date": game_date,
        "Home Team": home_team,
        "Away Team": away_team,
        "Home Rest Days": home_rest,
        "Away Rest Days": away_rest,
        "Home Game Number": home_game_number,
        "Home Win Probability": home_probability,
        "Away Win Probability": away_probability,
        "Predicted Winner": predicted_winner,
        "Prediction Confidence": confidence
    }

In [35]:
def predict_schedule(
    schedule,
    team_state_df,
    model
):
    predictions = []

    for _, game in schedule.iterrows():
        prediction = predict_game(
            home_team=game["HOME_TEAM"],
            away_team=game["AWAY_TEAM"],
            game_date=game["GAME_DATE"],
            schedule=schedule,
            team_state_df=team_state_df,
            model=model
        )

        predictions.append(
            prediction
        )

    return pd.DataFrame(predictions)

In [36]:
upcoming_predictions = predict_schedule(
    schedule,
    latest_team_state,
    production_model
)

upcoming_predictions

,Game Date,Home Team,Away Team,Home Rest Days,Away Rest Days,Home Game Number,Home Win Probability,Away Win Probability,Predicted Winner,Prediction Confidence
0,2026-10-20,Boston Celtics,Oklahoma City Thunder,3,3,1,0.494033,0.505967,Oklahoma City Thunder,0.505967
1,2026-10-20,Los Angeles Lakers,Golden State Warriors,3,3,1,0.616559,0.383441,Los Angeles Lakers,0.616559
2,2026-10-21,New York Knicks,Miami Heat,3,3,1,0.735697,0.264303,New York Knicks,0.735697


In [37]:
display_predictions = upcoming_predictions.copy()

probability_columns = [
    "Home Win Probability",
    "Away Win Probability",
    "Prediction Confidence"
]

for column in probability_columns:
    display_predictions[column] = (
        display_predictions[column]
        .map(lambda x: f"{x:.1%}")
    )

display_predictions

,Game Date,Home Team,Away Team,Home Rest Days,Away Rest Days,Home Game Number,Home Win Probability,Away Win Probability,Predicted Winner,Prediction Confidence
0,2026-10-20,Boston Celtics,Oklahoma City Thunder,3,3,1,49.4%,50.6%,Oklahoma City Thunder,50.6%
1,2026-10-20,Los Angeles Lakers,Golden State Warriors,3,3,1,61.7%,38.3%,Los Angeles Lakers,61.7%
2,2026-10-21,New York Knicks,Miami Heat,3,3,1,73.6%,26.4%,New York Knicks,73.6%


In [38]:
confidence_rankings = (
    upcoming_predictions
    .sort_values(
        "Prediction Confidence",
        ascending=False
    )
    .reset_index(drop=True)
)

confidence_rankings[
    [
        "Game Date",
        "Home Team",
        "Away Team",
        "Predicted Winner",
        "Prediction Confidence"
    ]
]

,Game Date,Home Team,Away Team,Predicted Winner,Prediction Confidence
0,2026-10-21,New York Knicks,Miami Heat,New York Knicks,0.735697
1,2026-10-20,Los Angeles Lakers,Golden State Warriors,Los Angeles Lakers,0.616559
2,2026-10-20,Boston Celtics,Oklahoma City Thunder,Oklahoma City Thunder,0.505967


## Validation

In [39]:
probability_sums = (
    upcoming_predictions[
        [
            "Home Win Probability",
            "Away Win Probability"
        ]
    ]
    .sum(axis=1)
)

print(
    "All probabilities sum to 1:",
    np.allclose(
        probability_sums,
        1.0
    )
)

All probabilities sum to 1: True


In [40]:
print(
    upcoming_predictions[
        "Home Win Probability"
    ].between(0, 1).all()
)

print(
    upcoming_predictions[
        "Away Win Probability"
    ].between(0, 1).all()
)

True
True


In [41]:
upcoming_predictions.isna().sum()

Game Date                0
Home Team                0
Away Team                0
Home Rest Days           0
Away Rest Days           0
Home Game Number         0
Home Win Probability     0
Away Win Probability     0
Predicted Winner         0
Prediction Confidence    0
dtype: int64

In [42]:
winner_check = np.where(
    upcoming_predictions[
        "Home Win Probability"
    ] >= 0.5,
    upcoming_predictions[
        "Home Team"
    ],
    upcoming_predictions[
        "Away Team"
    ]
)

print(
    "Winner labels valid:",
    np.array_equal(
        winner_check,
        upcoming_predictions[
            "Predicted Winner"
        ].values
    )
)

Winner labels valid: True


## Save Predictions and Production Model

In [43]:
upcoming_predictions.to_csv(
    "../data/processed/latest_predictions.csv",
    index=False
)

In [44]:
import joblib

In [45]:
from pathlib import Path

Path("../models").mkdir(
    exist_ok=True
)

In [46]:
joblib.dump(
    production_model,
    "../models/logistic_regression_production.pkl"
)

['../models/logistic_regression_production.pkl']

In [47]:
loaded_model = joblib.load(
    "../models/logistic_regression_production.pkl"
)

## Early-Season Considerations

The current live prediction pipeline initializes future matchups using the most recently available historical team state.

This creates a cold-start limitation at the beginning of a new season because current-season rolling statistics do not yet exist. Early predictions therefore rely heavily on the final statistical state of the previous season.

A future version of the pipeline will gradually blend prior-season information with current-season statistics as new games are completed. Once sufficient current-season games are available, rolling features can transition entirely to the new season.

This approach is intended to prevent extremely small early-season samples from disproportionately affecting predictions.

## Limitations

The current live prediction pipeline provides the architecture required for future NBA predictions but still has several limitations.

### Manual Schedule Input

Upcoming games are currently supplied manually rather than automatically retrieved from an NBA data source.

### Static Team State

Team statistics are initialized using the most recent historical information and are not yet automatically updated after new games are completed.

### Early-Season Cold Start

Rolling statistics from the new season are unavailable during the first several games. Prior-season statistics are therefore required as initial estimates of team strength.

### Player Availability

The current model does not include injuries, lineup changes, player availability, trades, or other roster-level information that may significantly affect individual games.

### Probability Uncertainty

Predicted probabilities represent estimated likelihoods rather than deterministic outcomes. Unexpected results remain possible even for high-confidence predictions.

## Production Pipeline Architecture

```text
Historical NBA Data
        ↓
Feature Engineering
        ↓
Production Training Dataset
        ↓
Logistic Regression
        ↓
Saved Production Model
        │
        ├──────── Team State
        │
        └──────── Upcoming Schedule
                        ↓
                Matchup Builder
                        ↓
                Rest-Day Features
                        ↓
                Win Probabilities
                        ↓
                Predicted Winner
                        ↓
              latest_predictions.csv
                        ↓
             Dashboard / Application


---

# 27. Final Conclusion

## Conclusion

Phase 6 transformed the historical NBA Prediction Engine into a future-facing prediction pipeline.

After model evaluation was completed in earlier phases, Logistic Regression was retrained using all available historical observations to create a production model. Reusable functions were developed to retrieve team state, construct future matchup features, calculate rest days and game numbers, generate game-level win probabilities, and process multiple upcoming games.

The pipeline produces structured predictions containing home and away win probabilities, predicted winners, and model confidence. Prediction outputs are saved independently from the modeling workflow so they can eventually be consumed by an interactive dashboard or external application.

The current implementation uses manual schedule input and historical team state. Future automation can replace these individual components without changing the underlying prediction architecture.

This phase completes the core NBA Prediction Engine by connecting historical data analysis, machine learning, explainability, simulation, and future-facing prediction into a unified system.